# FORCE Time Series Analysis (TSA)

### Purpose

The [Earth Observation Application Package (EOAP)](https://docs.ogc.org/bp/20-089r1.html) and surrounding tools implemented in the [`apex-force-openeo`](https://github.com/bcdev/apex-force-openeo) repository make it possible to use the [FORCE](https://force-eo.readthedocs.io/en/latest/) EO processing engine on the [Copernicus Data Space Ecosystem (CDSE)](https://dataspace.copernicus.eu/) using the [openEO](https://openeo.dataspace.copernicus.eu/) API.

> This notebook serves to showcase how to combine FORCE level 2 and FORCE Time Series Analysis (TSA) on CDSE openEO. For a more detailed discussion on how to run FORCE level 2, have a look at the [FORCE level 2 examples](../run-level2-and-download) or the [guide](https://esa-apex.github.io/apex_toolbox_documentation/docs/force/guide/intro.html).

### Prerequisites

In order to run this notebook, you need to have a CDSE account and sufficient openEO credits to run the jobs. Please note that CDSE provides a contigent of free credits each month, so you should be able to run the notebook if you haven't used up your credits with no additional cost.

In order to run FORCE TSA, you need to create a level 2 data cube first. Have a look at the [FORCE level2 process](https://algorithm-catalogue.apex.esa.int/apps/force_level2) first.

### Content

1. Connect to OpenEO backend
1. Run the FORCE Time Series Analysis (TSA) module on CDSE using the openEO API and the results from level 2 processing

### Citation and Acknowledgement

When using the FORCE processing engine, please make sure to properly [cite and acknowledge](https://force-eo.readthedocs.io/en/latest/policy/citation.html)
the software and its related [scientific publications](https://force-eo.readthedocs.io/en/latest/refs.html#refs) .
The implementation as EOAP was performed in the context of ESA's [APEx](https://apex.esa.int/) initiative.

### Further reading

**FORCE on CDSE using OpenEO**

1. [User guide](https://esa-apex.github.io/apex_toolbox_documentation/docs/force/guide/intro.html)
2. [Parameters](https://esa-apex.github.io/apex_toolbox_documentation/docs/force/guide/parametrization.html)

**APEx**

1. [Project website](https://apex.esa.int/)
2. [Documentation Portal](https://esa-apex.github.io/apex_documentation/)

**FORCE**

1. [Documentation](https://force-eo.readthedocs.io/en/latest/index.html)
1. [Level 2 processing system](https://force-eo.readthedocs.io/en/latest/components/lower-level/level2/)
1. [Citation and acknowledgement](https://force-eo.readthedocs.io/en/latest/policy/citation.html)


**openEO**

1. [openEO on CDSE](https://documentation.dataspace.copernicus.eu/APIs/openEO/openEO.html)
1. [Credit Usage on CDSE](https://documentation.dataspace.copernicus.eu/APIs/openEO/credit_usage.html)
1. [openEO project website](https://openeo.org/)
1. [openEO Python Client documentation](https://open-eo.github.io/openeo-python-client/)

### Imports

In [ ]:
from pathlib import Path
import json
import getpass
import time

import openeo
import pystac
import pystac_client

### The FORCE level 2 data cube as input

The Time Series Analysis requires a FORCE level 2 data cube as input. You can use the [FORCE level2 process](https://algorithm-catalogue.apex.esa.int/apps/force_level2) to generate one.

When you have generated a level 2 cube, remember the merge path you specified for saving the cube to the `apex-force-results-workspace` (or any other) workspace.

In [ ]:
LEVEL2_MERGE=None # <---- replace with your merge path!
TSA_MERGE=None    # <---- replace with your merge path!

With the merge, we can find the `catalog.json` generated by FORCE level2. You can paste the url into your web browser to download the STAC document.

In [ ]:
WORKSPACE_URL = "https://s3.waw4-1.cloudferro.com/apex-force-results-waw4-1-exotc5yuexi2c5tvwqhoivj62fz8v0uupy0me"
l2_catalog_url = f"{WORKSPACE_URL}/{LEVEL2_MERGE}/catalog.json"
print(l2_catalog_url)

### Connect to the OpenEO backend

In [ ]:
backend_url = "openeo.dataspace.copernicus.eu"
connection = openeo.connect(backend_url).authenticate_oidc()

### Tiles

When processing in the cloud, we cannot just inspect the processed datacube in the local file system. Instead, we can inspect the STAC catalog produced by the FORCE level 2 process to gather information about the resulting cube.
The processed tiles (in the sense of the [FORCE datacube model](https://force-eo.readthedocs.io/en/latest/howto/datacube.html)) can be determined from the assets present in the STAC Item. 

This can be done in the browser (replace the `catalog.json` part of the catalog URL with the name of your STAC item (listed in the catalog).
More comfortably, you can do this programmatically, as shown in the [long form example for TSA](https://github.com/bcdev/apex-force-openeo/blob/main/examples/run-level2-and-tsa/FORCE_level2_and_TSA.ipynb).

```Python
# not included here, works only outside Jupyterlite

l2_catalog = pystac.Catalog.from_file(l2_catalog_url)
l2_item = next(l2_catalog.get_items())

tiles = set([k.split(".")[0] for k in l2_item.assets.keys() if k.endswith("BOA")])
tile_x = set(int(tile[1:5]) for tile in tiles)
tile_y = set(int(tile[7:]) for tile in tiles)
print(tile_x, tile_y)
```

This information can be useful to set the `x_tile_range` and `y_tile_range` parameters for higher level processing, for example:

In [ ]:
x_tile_range = [30, 31] # <---- adjust to your own level 2 example!
y_tile_range = [29, 29] # <---- adjust to your own level 2 example!
temporal_extent = ["2026-04-17", "2026-04-18"] # <---- adjust to your own level 2 example!

## Running Force Time Series Analysis (TSA)

After level 2 processing, we can use the [FORCE TSA module](https://force-eo.readthedocs.io/en/latest/components/higher-level/tsa/) to generate pixel wise time series statistics (for example). The level 2 datacube is passed to the TSA module as a URL pointing to the STAC catalog (or Item) extracted in the last step.

You may want to test the configuration for the TSA module on a small subset of the tiles, to make sure you are getting the correct results. You can run FORCE TSA multiple times on the same input with different configurations by passing the same level 2 catalog to the process.

For more details on setting parameters for FORCE TSA, check out [the documentation](https://esa-apex.github.io/apex_toolbox_documentation/docs/force/guide/parametrization.html).

In [ ]:
user = getpass.getuser() # running on your machine? Your username is used as merge. It may not be unique, so better choose a unique name.
if user == "web_user": # running via jupyterlite? Please set the merge by hand!
    user = None
    # TSA_MERGE = "unique-string"  # <--- set our own merge path

processing_name_tsa = "FORCE_TSA_example"

# Be careful: The merge determines the storage location of your results on the publicly accessible apex-force-results-workspace.
# You may overwrite your results or those of others, if the `merge` parameter is not
# Set to a unique value.
if not TSA_MERGE and user is not None:
    TSA_MERGE = f"{user}/experiments/{processing_name_tsa}"
    

In [ ]:
if LEVEL2_MERGE is None or TSA_MERGE is None:
    raise RuntimeError(
        "Please set LEVEL2_MERGE to the workspace merge path of your level 2 cube "
        "and TSA_MERGE with the merge path where you would like to store your results (see above)."
    )




stac_resource = openeo.rest.stac_resource.StacResource(
    graph=openeo.internal.graph_building.PGNode(
        process_id="force_tsa",
        arguments={
            "stac_url": l2_catalog_url,
            "name": processing_name_tsa,
            "date_range": temporal_extent,
            "x_tile_range": x_tile_range,
            "y_tile_range": y_tile_range,
            "stm": ["AVG"],
            "output_stm": True,
            # <---- more parameters for FORCE TSA
        },
    ),
    connection=connection,
)
stac_resource = stac_resource.export_workspace(
    workspace="apex-force-results-workspace",
    merge=TSA_MERGE,
)
tsa_job = stac_resource.create_job(title=f"{processing_name_tsa}: {TSA_MERGE}", job_options = {"export-workspace-enable-merge": False}))
tsa_job.start_and_wait()

## Result access

If you included the `export_workspace` step above in the process graph, the results are publicly availble on the apex-force-results workspace, you may use any S3 tools to download them. For example, use [`s5cmd`](https://github.com/peak/s5cmd):

```bash
# Downloads the FORCE data cube to the current working directory:
# Replace <merge> with your merge path.

s5cmd --endpoint-url https://s3.waw4-1.cloudferro.com/ --no-sign-request cp s3://apex-force-results-waw4-1-exotc5yuexi2c5tvwqhoivj62fz8v0uupy0me/<merge>/* .
```
For more and more detailed examples, see <https://github.com/bcdev/apex-force-openeo/tree/main/examples>


In [ ]:
tsa_results = tsa_job.get_results()
tsa_results